In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.io as pio
import plotly.express as px
import squarify
import os

# Caminho de saída dos HTMLs — ajusta conforme o teu projeto
OUTPUT_DIR = "./assets/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

dataset = pd.read_csv("./DATASET/dataset_limpo_medianag.csv")

In [20]:
#G1: Seniority Distribution (Donut)
df_g2 = dataset.copy()
df_g2["YearsCodePro"] = pd.to_numeric(
    df_g2["YearsCodePro"].replace({"Less than 1 year": 0, "More than 50 years": 51}),
    errors="coerce")

def seniority(y):
    if pd.isna(y):  return None
    if y < 2:       return "Junior"
    if y < 6:       return "Mid"
    if y < 12:      return "Senior"
    return "Expert"

df_g2["Seniority"] = df_g2["YearsCodePro"].apply(seniority)
df_g2 = df_g2[df_g2["Seniority"].notna()]

ordem     = ["Junior", "Mid", "Senior", "Expert"]
cores_sen = ["#1e88e5", "#14b8a6", "#fbbf24", "#fb7f3f"]
sen_counts = df_g2["Seniority"].value_counts().reindex(ordem).fillna(0).reset_index()
sen_counts.columns = ["Seniority", "Count"]

fig = go.Figure(go.Pie(
    labels=sen_counts["Seniority"],
    values=sen_counts["Count"],
    hole=0.5,
    marker=dict(colors=cores_sen, line=dict(color="white", width=2.5)),
    textinfo="label+percent",
    textfont=dict(size=12, color="white")
))

fig.add_annotation(text="Seniority", x=0.5, y=0.5,
                   font=dict(size=14, color="black"), showarrow=False)

fig.write_html(OUTPUT_DIR + "g1_seniority.html")
fig.show(renderer="browser")

In [21]:
#G2: Age Distribution (Barras horizontais)
ordem_age   = ["<18", "18-24", "25-34", "35-44", "45-54", "55-64", "65+"]
geracao_map = {"<18":"Gen Alpha", "18-24":"Gen Z", "25-34":"Millennials",
               "35-44":"Gen X", "45-54":"Boomers", "55-64":"Boomers", "65+":"Silent"}
cores_ger   = {"Gen Alpha":"#7c78d1", "Gen Z":"#1e88e5", "Millennials":"#14b8a6",
               "Gen X":"#fbbf24", "Boomers":"#fb7f3f", "Silent":"#64748b"}

age_counts = (dataset["Age"].dropna()
              .replace("Sem dado", None).dropna()
              .value_counts().reindex(ordem_age).fillna(0).reset_index())
age_counts.columns    = ["Age", "Count"]
age_counts["Percent"] = (age_counts["Count"] / len(dataset)) * 100
age_counts["Geração"] = age_counts["Age"].map(geracao_map)
age_counts["Color"]   = age_counts["Geração"].map(cores_ger)

fig = px.bar(
    age_counts, x="Percent", y="Age", orientation="h",
    color="Geração",
    color_discrete_map=cores_ger,
    text=age_counts.apply(lambda r: f'{r["Percent"]:.1f}% — {r["Geração"]}', axis=1),
    labels={"Percent": "% of Respondents", "Age": "Age Group"}
)
fig.update_traces(textposition="outside")
fig.update_yaxes(categoryorder="array", categoryarray=list(reversed(ordem_age)))
fig.update_layout(
    margin=dict(l=20, r=20, t=60, b=20),
    xaxis=dict(showgrid=False),
    plot_bgcolor="white"
)

fig.write_html(OUTPUT_DIR + "g2_age.html")
fig.show(renderer="browser")

In [5]:
df_g1 = dataset[dataset["ConvertedCompYearly"] != 65000]
df_g1 = df_g1[["DevType", "ConvertedCompYearly"]].dropna()
df_g1 = df_g1.assign(DevType=df_g1["DevType"].str.split(";")).explode("DevType")
df_g1["DevType"] = df_g1["DevType"].str.strip().replace("Sem dado", None)
df_g1 = df_g1.dropna()
df_g1 = df_g1[~df_g1["DevType"].str.startswith("Other (")]

top_devtypes = df_g1["DevType"].value_counts().head(10).index
df_g1 = df_g1[df_g1["DevType"].isin(top_devtypes)]

cap = df_g1["ConvertedCompYearly"].quantile(0.95)
df_g1 = df_g1[df_g1["ConvertedCompYearly"] <= cap]

ordem = df_g1.groupby("DevType")["ConvertedCompYearly"].median().sort_values().index

colors_box = ['#1e88e5','#14b8a6','#fbbf24','#fb7f3f','#7c78d1',
              '#84c69b','#e53935','#8d6e63','#26a69a','#5e35b1']

fig = go.Figure()

for dev, color in zip(ordem, colors_box):
    vals = df_g1[df_g1["DevType"] == dev]["ConvertedCompYearly"]
    fig.add_trace(go.Box(
        x=vals,
        name=dev,
        orientation="h",
        fillcolor=color,
        opacity=0.7,
        line=dict(color="#e53935", width=2),
        marker=dict(color=color)
    ))

fig.update_layout(
    title=dict(text="Salary Distribution by Developer Type", font=dict(size=14)),
    xaxis_title="Annual Salary (USD)",
    yaxis_title="",
    showlegend=False,
    plot_bgcolor="white",
    bargap=0.3
)

fig.update_xaxes(showgrid=True, gridcolor="#eeeeee")
fig.update_yaxes(showgrid=False)

fig.write_html(OUTPUT_DIR + "g3_salary_devtype.html")
fig.show(renderer="browser")

In [23]:
#G4: Median Salary by Experience (Line)
df_g3 = dataset[dataset["ConvertedCompYearly"] != 65000]
df_g3 = df_g3[["YearsCodePro", "ConvertedCompYearly"]].dropna()
df_g3["YearsCodePro"] = pd.to_numeric(
    df_g3["YearsCodePro"].replace({"Less than 1 year": 0, "More than 50 years": 51}),
    errors="coerce")
df_g3 = df_g3.dropna()
cap   = df_g3["ConvertedCompYearly"].quantile(0.95)
df_g3 = df_g3[df_g3["ConvertedCompYearly"] <= cap]

bins   = [0, 2, 5, 10, 15, 20, 30, 51]
labels = ["0-2", "3-5", "6-10", "11-15", "16-20", "21-30", "30+"]
df_g3["GrupoExp"] = pd.cut(df_g3["YearsCodePro"], bins=bins, labels=labels, right=True)
mediana_grupo = df_g3.groupby("GrupoExp", observed=True)["ConvertedCompYearly"].median().reset_index()
mediana_grupo.columns = ["GrupoExp", "MedianSalary"]
mediana_grupo["MedianSalary_K"] = mediana_grupo["MedianSalary"] / 1000

fig = px.line(
    mediana_grupo, x="GrupoExp", y="MedianSalary_K",
    markers=True,
    labels={"GrupoExp": "Years of Professional Experience", "MedianSalary_K": "Median Salary (USD K)"},
    text=mediana_grupo["MedianSalary_K"].apply(lambda v: f"${v:.0f}K")
)
fig.update_traces(
    line=dict(color="#1e88e5", width=2.5),
    marker=dict(color="#e53935", size=9),
    textposition="top center"
)
fig.update_layout(
    plot_bgcolor="white",
    margin=dict(l=20, r=20, t=60, b=20)
)

fig.write_html(OUTPUT_DIR + "g4_salary_exp.html")
fig.show(renderer="browser")

In [24]:
#G5: Most Used Tools (Barras horizontais)
tools_counts = (
    dataset["ToolsTechHaveWorkedWith"]
    .dropna().str.split(";").explode().str.strip()
    .replace("Sem dado", None).dropna()
    .value_counts().head(15).reset_index()
)
tools_counts.columns    = ["Tool", "Count"]
tools_counts["Percent"] = (tools_counts["Count"] / len(dataset)) * 100

fig = px.bar(
    tools_counts.sort_values("Percent"),
    x="Percent", y="Tool", orientation="h",
    text=tools_counts.sort_values("Percent")["Percent"].apply(lambda v: f"{v:.1f}%"),
    labels={"Percent": "% of Respondents", "Tool": ""},
    color_discrete_sequence=["#14b8a6"]
)
fig.update_traces(textposition="outside", marker_color="#14b8a6")
fig.update_layout(
    plot_bgcolor="white",
    margin=dict(l=20, r=20, t=60, b=20)
)

fig.write_html(OUTPUT_DIR + "g5_tools.html")
fig.show(renderer="browser")

In [25]:
#G6: Remote Work (Donut)
remote_counts = (
    dataset["RemoteWork"].dropna()
    .replace("Sem dado", None).dropna()
    .value_counts().reset_index()
)
remote_counts.columns    = ["RemoteWork", "Count"]
remote_counts["Percent"] = (remote_counts["Count"] / remote_counts["Count"].sum()) * 100

fig = go.Figure(go.Pie(
    labels=remote_counts["RemoteWork"],
    values=remote_counts["Count"],
    hole=0.5,
    marker=dict(colors=["#1e88e5", "#14b8a6", "#fbbf24"],
                line=dict(color="white", width=2.5)),
    textinfo="label+percent",
    textfont=dict(size=12)
))
fig.add_annotation(text="Remote<br>Work", x=0.5, y=0.5,
                   font=dict(size=13, color="black"), showarrow=False)
fig.update_layout(
    margin=dict(l=20, r=20, t=60, b=20)
)

fig.write_html(OUTPUT_DIR + "g6_remote.html")
fig.show(renderer="browser")

In [26]:
#G7: Job Satisfaction Factors (Bar horizontal)
jobsat_cols = {
    "JobSatPoints_1":  "Industry",
    "JobSatPoints_4":  "Management",
    "JobSatPoints_5":  "Languages",
    "JobSatPoints_6":  "Compensation",
    "JobSatPoints_7":  "Flexibility",
    "JobSatPoints_8":  "Environment",
    "JobSatPoints_9":  "Diversity",
    "JobSatPoints_10": "Mission/Values",
    "JobSatPoints_11": "Learning",
}

medias = {}
for col, label in jobsat_cols.items():
    if col in dataset.columns:
        valores = pd.to_numeric(dataset[col], errors="coerce").dropna()
        medias[label] = valores.mean()

medias_df = pd.Series(medias).sort_values(ascending=True).reset_index()
medias_df.columns = ["Fator", "Media"]

fig = px.bar(
    medias_df, x="Media", y="Fator", orientation="h",
    text=medias_df["Media"].apply(lambda v: f"{v:.1f} pts"),
    labels={"Media": "Average Score", "Fator": ""},
    color_discrete_sequence=["#1e3a5f"]
)
fig.update_traces(textposition="outside")
fig.update_layout(
    plot_bgcolor="white",
    margin=dict(l=20, r=20, t=60, b=20)
)

fig.write_html(OUTPUT_DIR + "g7_jobsat.html")
fig.show(renderer="browser")

In [27]:
# G8: Industries (Treemap)
industry_counts = (
    dataset["Industry"].dropna()
    .replace("Sem dado", None).dropna()
    .value_counts().head(12).reset_index()
)
industry_counts.columns    = ["Industry", "Count"]
industry_counts["Percent"] = (industry_counts["Count"] / len(dataset)) * 100
industry_counts["Label"]   = industry_counts.apply(
    lambda r: f'{r["Industry"]}<br>{r["Percent"]:.1f}%', axis=1)

fig = px.treemap(
    industry_counts,
    path=["Industry"], values="Count",
    color="Count",
    color_continuous_scale=["#1e88e5","#14b8a6","#fbbf24","#fb7f3f","#7c78d1"],
    custom_data=["Percent"]
)
fig.update_traces(
    texttemplate="%{label}<br>%{customdata[0]:.1f}%",
    textfont=dict(size=13)
)
fig.update_layout(
    margin=dict(l=20, r=20, t=60, b=20)
)

fig.write_html(OUTPUT_DIR + "g8_industries.html")
fig.show(renderer="browser")

In [28]:
#G9: Mapa Mundial (Choropleth)
nome_map = {
    "United Kingdom of Great Britain and Northern Ireland": "United Kingdom",
    "Viet Nam": "Vietnam", "Russian Federation": "Russia",
    "Republic of Korea": "South Korea", "Republic of Moldova": "Moldova",
    "Iran, Islamic Republic of...": "Iran", "Venezuela, Bolivarian Republic of...": "Venezuela",
    "Hong Kong (S.A.R.)": "Hong Kong", "Sem dado": None, "Nomadic": None,
}

df_mapa = dataset[dataset["ConvertedCompYearly"] != 65000]
df_mapa = df_mapa[["Country", "ConvertedCompYearly"]].dropna()
df_mapa["Country"] = df_mapa["Country"].replace(nome_map)
df_mapa = df_mapa[df_mapa["Country"].notna()]

paises_validos = df_mapa["Country"].value_counts()
paises_validos = paises_validos[paises_validos >= 50].index
df_mapa = df_mapa[df_mapa["Country"].isin(paises_validos)]

mediana_pais = df_mapa.groupby("Country")["ConvertedCompYearly"].median().reset_index()
mediana_pais.columns      = ["Country", "Mediana"]
mediana_pais["Mediana_K"] = (mediana_pais["Mediana"] / 1000).round(1)

fig = px.choropleth(
    mediana_pais, locations="Country", locationmode="country names",
    color="Mediana_K", color_continuous_scale="Blues",
    labels={"Mediana_K": "Median Salary (USD K)"},
    hover_name="Country"
)
fig.update_layout(
    geo=dict(showframe=False, showcoastlines=True),
    margin=dict(l=0, r=0, t=60, b=0)
)

fig.write_html(OUTPUT_DIR + "g9_map.html")
fig.show(renderer="browser")

C:\Users\marco\AppData\Local\Temp\ipykernel_21720\1923880849.py:23: DeprecationWarning: The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.
  fig = px.choropleth(


In [29]:
#G10: Most Used Development Tools

tools_counts = (
    dataset["ToolsTechHaveWorkedWith"]
    .dropna()
    .str.split(";")
    .explode()
    .str.strip()
    .replace("Sem dado", None)
    .dropna()
    .value_counts()
    .head(15)
    .reset_index()
)
tools_counts.columns    = ["Tool", "Count"]
tools_counts["Percent"] = (tools_counts["Count"] / len(dataset)) * 100

import plotly.express as px

fig = px.bar(
    tools_counts,
    x="Percent",
    y="Tool",
    orientation="h",
    text=tools_counts["Percent"].map(lambda x: f"{x:.1f}%"),
    color_discrete_sequence=["#14b8a6"]
)

fig.update_layout(
    xaxis_title="% of Respondents",
    yaxis=dict(autorange="reversed"),
)

fig.write_html(OUTPUT_DIR + "g10_tools.html")
fig.show(renderer="browser")

In [30]:
#G11: Remote Work Preference (Donut)
remote_counts = (
    dataset["RemoteWork"]
    .dropna()
    .replace("Sem dado", None)
    .dropna()
    .value_counts()
    .reset_index()
)

remote_counts.columns = ["RemoteWork", "Count"]
remote_counts["Percent"] = (remote_counts["Count"] / remote_counts["Count"].sum()) * 100

import plotly.express as px

fig = px.pie(
    remote_counts,
    names="RemoteWork",
    values="Percent",
    hole=0.5,
    color_discrete_sequence=["#1e88e5", "#14b8a6", "#fbbf24"]
)

fig.update_traces(
    textinfo='percent+label',
    textfont_size=12
)

fig.update_layout(
    annotations=[dict(text="Remote<br>Work", x=0.5, y=0.5, font_size=14, showarrow=False)]
)

fig.write_html(OUTPUT_DIR + "g11_remote_work.html")
fig.show(renderer="browser")

In [31]:
#G12: Job Satisfaction Factors (Bar horizontal)
jobsat_cols = {
    "JobSatPoints_1":  "Industry",
    "JobSatPoints_4":  "Management",
    "JobSatPoints_5":  "Languages",
    "JobSatPoints_6":  "Compensation",
    "JobSatPoints_7":  "Flexibility",
    "JobSatPoints_8":  "Environment",
    "JobSatPoints_9":  "Diversity",
    "JobSatPoints_10": "Mission/Values",
    "JobSatPoints_11": "Learning",
}

medias = {}
for col, label in jobsat_cols.items():
    if col in dataset.columns:
        valores = pd.to_numeric(dataset[col], errors="coerce").dropna()
        medias[label] = valores.mean()

medias_df = pd.Series(medias).sort_values(ascending=False).reset_index()
medias_df.columns = ["Fator", "Media"]

max_val = medias_df["Media"].max()
medias_df["Pct"] = (medias_df["Media"] / max_val) * 100

import plotly.express as px

fig = px.bar(
    medias_df,
    x="Pct",
    y="Fator",
    orientation="h",
    text=medias_df["Media"].map(lambda x: f"{x:.1f} pts"),
    color_discrete_sequence=["#1e3a5f"]
)

fig.update_layout(
    xaxis_title="Relative Importance (%)",
    yaxis_title="",
    yaxis=dict(autorange="reversed"),
    plot_bgcolor="#ffffff",
    paper_bgcolor="#ffffff",
    annotations=[
        dict(
            text="<b>Insight:</b> Compensation and Flexibility are the most valued factors,<br>"
                 "suggesting salary and work-life balance are top priorities.",
            x=0.5,
            y=-0.25,
            xref="paper",
            yref="paper",
            showarrow=False,
            font=dict(size=11, color="#1e3a5f"),
            align="center"
        )
    ]
)

fig.write_html(OUTPUT_DIR + "g12_jobsat_metrics.html")
fig.show(renderer="browser")

In [32]:
#G13: Industries 

import plotly.express as px

industry_counts = (
    dataset["Industry"]
    .dropna()
    .replace("Sem dado", None)
    .dropna()
    .value_counts()
    .head(12)
    .reset_index()
)

industry_counts.columns = ["Industry", "Count"]

# 👇 percentagem correta (igual ao teu código original)
industry_counts["Percent"] = (industry_counts["Count"] / len(dataset)) * 100

# 👇 criar label já com percentagem (evita bugs do plotly)
industry_counts["Label"] = industry_counts.apply(
    lambda x: f'{x["Industry"]}<br>{x["Percent"]:.1f}%', axis=1
)

cores = [
    "#1e88e5","#14b8a6","#fbbf24","#fb7f3f","#7c78d1",
    "#84c69b","#e53935","#8d6e63","#26a69a","#5e35b1","#f4511e","#039be5"
]

fig = px.treemap(
    industry_counts,
    path=["Label"],        # 👈 usar label já pronto
    values="Count",
    color="Industry",
    color_discrete_sequence=cores
)

fig.update_layout(
)

fig.write_html(OUTPUT_DIR + "g13_industries_treemap.html")
fig.show(renderer="browser")

In [2]:
devtype_counts = (
    dataset["DevType"]
    .dropna()
    .str.split(";")
    .explode()
    .str.strip()
    .replace("Sem dado", None)
    .dropna()
    .pipe(lambda s: s[~s.str.startswith("Other")])
    .value_counts()
    .head(15)
    .reset_index()
)
devtype_counts.columns = ["DevType", "Count"]
devtype_counts["Percent"] = (devtype_counts["Count"] / devtype_counts["Count"].sum() * 100).round(1)

fig = px.bar(
    devtype_counts.sort_values("Percent"),
    x="Percent",
    y="DevType",
    orientation="h",
    text="Percent",
    color_discrete_sequence=["#1e88e5"]
)

fig.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside",
    marker_line_color="white",
    marker_line_width=1
)

fig.update_layout(
    title=dict(text="Types of Developers in the Market", font=dict(size=14)),
    xaxis_title="% of Respondents",
    yaxis_title="",
    xaxis=dict(range=[0, devtype_counts["Percent"].max() + 6]),
    bargap=0.35,
    plot_bgcolor="white",
    showlegend=False
)

fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)

fig.write_html(OUTPUT_DIR + "g14_devtypes.html")
fig.show(renderer="browser")

In [34]:
#G15: AI Tool Adoption Among Developers

import plotly.graph_objects as go

ai_counts = (
    dataset["AISelect"]
    .dropna()
    .replace("Sem dado", None)
    .dropna()
    .value_counts()
    .reset_index()
)
ai_counts.columns = ["AISelect", "Count"]
ai_counts["Percent"] = (ai_counts["Count"] / ai_counts["Count"].sum() * 100).round(1)

# Labels mais curtos
label_map = {
    "Yes": "Uses AI",
    "No, but I plan to soon": "Plans to Use",
    "No, and I don't plan to": "No Plans"
}
ai_counts["Label"] = ai_counts["AISelect"].map(label_map).fillna(ai_counts["AISelect"])

cores = ["#1e88e5", "#14b8a6", "#fbbf24"]

fig = go.Figure(data=[go.Pie(
    labels=ai_counts["Label"],
    values=ai_counts["Percent"],
    hole=0.55,
    marker=dict(colors=cores, line=dict(color="white", width=2.5)),
    textinfo="label+percent",
    textfont=dict(size=13, color="white"),
    hovertemplate="<b>%{label}</b><br>%{value}% of respondents<extra></extra>",
)])

fig.update_layout(
    annotations=[dict(
        text="AI<br>Adoption",
        x=0.5, y=0.5,
        font=dict(size=14, color="#0f172a", family="Arial Black"),
        showarrow=False
    )],
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.15,
        xanchor="center",
        x=0.5,
        font=dict(size=12)
    ),
    paper_bgcolor="white",
    plot_bgcolor="white",
    margin=dict(t=80, b=80, l=20, r=20),
    width=600,
    height=500
)

# Salvar em HTML
fig.write_html(OUTPUT_DIR + "g15_ai_adoption.html")
fig.show(renderer="browser")

In [35]:
#G16: Most Used AI Tools Among Developers (corrigido definitivo)

import plotly.graph_objects as go

# Explodir, limpar e remover "Other"
ai_tools = (
    dataset["AIToolCurrently Using"]
    .dropna()
    .str.split(";")
    .explode()
    .str.strip()
    .replace("Sem dado", None)
)

# Criar máscara segura para descartar nulos e "Other"
mask = ai_tools.notna() & (~ai_tools.fillna("").str.startswith("Other"))
ai_tools = ai_tools[mask]

# Contar e pegar top 15
ai_tools = ai_tools.value_counts().head(15).reset_index()
ai_tools.columns = ["Tool", "Count"]

# Percentuais
ai_tools["Percent"] = (ai_tools["Count"] / ai_tools["Count"].sum() * 100).round(1)
ai_tools = ai_tools.sort_values("Percent", ascending=True)

# Plot
fig = go.Figure(go.Bar(
    x=ai_tools["Percent"],
    y=ai_tools["Tool"],
    orientation="h",
    marker=dict(
        color=ai_tools["Percent"],
        colorscale="Blues",
        showscale=True,
        colorbar=dict(title="% Respondents")
    ),
    text=[f'{p:.1f}%' for p in ai_tools["Percent"]],
    textposition="outside",
    hovertemplate="<b>%{y}</b><br>%{x:.1f}% of respondents<extra></extra>",
))

fig.update_layout(
    xaxis=dict(title="% of Respondents", showgrid=True, gridcolor="#e5e7eb", zeroline=False),
    yaxis=dict(title="", tickfont=dict(size=11)),
    paper_bgcolor="white",
    plot_bgcolor="white",
    margin=dict(t=80, b=60, l=20, r=80),
    width=800,
    height=550,
)

# Salvar em HTML
fig.write_html(OUTPUT_DIR + "g16_ai_tools.html")
fig.show(renderer="browser")

In [36]:
#Novos graficos (G17-G25) 

In [37]:
#G17: AI Search & Dev Tools
ai_search = (
    dataset["AISearchDevHaveWorkedWith"]
    .dropna()
    .str.split(";")
    .explode()
    .str.strip()
    .replace("Sem dado", None)
    .dropna()
    .value_counts()
    .head(10)
    .reset_index()
)
ai_search.columns    = ["Tool", "Count"]
ai_search["Percent"] = (ai_search["Count"] / len(dataset) * 100).round(1)

fig = go.Figure(go.Bar(
    x=ai_search["Tool"],
    y=ai_search["Percent"],
    marker=dict(
        color=ai_search["Percent"],
        colorscale="Teal",
        showscale=False,
        line=dict(color="white", width=1.5)
    ),
    text=[f'{p:.1f}%' for p in ai_search["Percent"]],
    textposition="outside",
    hovertemplate="<b>%{x}</b><br>%{y:.1f}%<extra></extra>",
))
fig.update_layout(
    xaxis=dict(tickangle=-30, tickfont=dict(size=10), showgrid=False),
    yaxis=dict(
        title="% Respondents",
        showgrid=True,
        gridcolor="#e5e7eb",
        zeroline=False,
        range=[0, ai_search["Percent"].max() + 8]
    ),
    paper_bgcolor="white",
    plot_bgcolor="white",
    margin=dict(t=60, b=80, l=50, r=20),
)

fig.write_html(OUTPUT_DIR + "g17_ai_search.html")
fig.show(renderer="browser")

In [38]:
#G18: Most Used Programming Languages
lang_counts = (
    dataset["LanguageHaveWorkedWith"]
    .dropna()
    .str.split(";").explode().str.strip()
    .replace("Sem dado", None).dropna()
    .value_counts().head(15).reset_index()
)
lang_counts.columns    = ["Language", "Count"]
lang_counts["Percent"] = (lang_counts["Count"] / len(dataset) * 100).round(1)
lang_counts = lang_counts.sort_values("Percent", ascending=True)

fig = go.Figure(go.Bar(
    x=lang_counts["Percent"],
    y=lang_counts["Language"],
    orientation="h",
    marker=dict(color=lang_counts["Percent"], colorscale="Blues", showscale=False),
    text=[f'{p:.1f}%' for p in lang_counts["Percent"]],
    textposition="outside",
    hovertemplate="<b>%{y}</b><br>%{x:.1f}%<extra></extra>",
))
fig.update_layout(
    xaxis=dict(
        title="% of Respondents",
        showgrid=True,
        gridcolor="#e5e7eb",
        range=[0, lang_counts["Percent"].max() + 8]
    ),
    yaxis=dict(tickfont=dict(size=10)),
    paper_bgcolor="white",
    plot_bgcolor="white",
    margin=dict(t=60, b=40, l=20, r=80),
)

fig.write_html(OUTPUT_DIR + "g18_languages.html")
fig.show(renderer="browser")

In [39]:
#G19: Top Web Frameworks Used
web_counts = (
    dataset["WebframeHaveWorkedWith"]
    .dropna()
    .str.split(";").explode().str.strip()
    .replace("Sem dado", None).dropna()
    .value_counts().head(12).reset_index()
)
web_counts.columns    = ["Framework", "Count"]
web_counts["Percent"] = (web_counts["Count"] / len(dataset) * 100).round(1)

fig = go.Figure(go.Bar(
    x=web_counts["Framework"],
    y=web_counts["Percent"],
    marker=dict(
        color=web_counts["Percent"],
        colorscale="Teal",
        showscale=False,
        line=dict(color="white", width=1.5)
    ),
    text=[f'{p:.1f}%' for p in web_counts["Percent"]],
    textposition="outside",
    hovertemplate="<b>%{x}</b><br>%{y:.1f}%<extra></extra>",
))
fig.update_layout(
    xaxis=dict(tickangle=-30, tickfont=dict(size=10), showgrid=False),
    yaxis=dict(
        title="% of Respondents",
        showgrid=True,
        gridcolor="#e5e7eb",
        range=[0, web_counts["Percent"].max() + 8]
    ),
    paper_bgcolor="white",
    plot_bgcolor="white",
    margin=dict(t=60, b=80, l=50, r=20),
)

fig.write_html(OUTPUT_DIR + "g19_web_frameworks.html")
fig.show(renderer="browser")

In [40]:
#G20: Top Databases Used
db_counts = (
    dataset["DatabaseHaveWorkedWith"]
    .dropna()
    .str.split(";").explode().str.strip()
    .replace("Sem dado", None).dropna()
    .value_counts().head(12).reset_index()
)
db_counts.columns    = ["Database", "Count"]
db_counts["Percent"] = (db_counts["Count"] / len(dataset) * 100).round(1)
db_counts = db_counts.sort_values("Percent", ascending=True)

fig = go.Figure(go.Bar(
    x=db_counts["Percent"],
    y=db_counts["Database"],
    orientation="h",
    marker=dict(color=db_counts["Percent"], colorscale="Oranges", showscale=False),
    text=[f'{p:.1f}%' for p in db_counts["Percent"]],
    textposition="outside",
    hovertemplate="<b>%{y}</b><br>%{x:.1f}%<extra></extra>",
))
fig.update_layout(
    xaxis=dict(
        title="% of Respondents",
        showgrid=True,
        gridcolor="#e5e7eb",
        range=[0, db_counts["Percent"].max() + 8]
    ),
    yaxis=dict(tickfont=dict(size=10)),
    paper_bgcolor="white",
    plot_bgcolor="white",
    margin=dict(t=60, b=40, l=20, r=80),
)

fig.write_html(OUTPUT_DIR + "g20_databases.html")
fig.show(renderer="browser")

In [41]:
#G21: Top IDEs & Code Editors Used
ide_counts = (
    dataset["NEWCollabToolsHaveWorkedWith"]
    .dropna()
    .str.split(";").explode().str.strip()
    .replace("Sem dado", None).dropna()
    .value_counts().head(12).reset_index()
)
ide_counts.columns    = ["IDE", "Count"]
ide_counts["Percent"] = (ide_counts["Count"] / len(dataset) * 100).round(1)

fig = go.Figure(go.Bar(
    x=ide_counts["IDE"],
    y=ide_counts["Percent"],
    marker=dict(
        color=ide_counts["Percent"],
        colorscale="Purples",
        showscale=False,
        line=dict(color="white", width=1.5)
    ),
    text=[f'{p:.1f}%' for p in ide_counts["Percent"]],
    textposition="outside",
    hovertemplate="<b>%{x}</b><br>%{y:.1f}%<extra></extra>",
))
fig.update_layout(
    xaxis=dict(tickangle=-30, tickfont=dict(size=10), showgrid=False),
    yaxis=dict(
        title="% of Respondents",
        showgrid=True,
        gridcolor="#e5e7eb",
        range=[0, ide_counts["Percent"].max() + 8]
    ),
    paper_bgcolor="white",
    plot_bgcolor="white",
    margin=dict(t=60, b=80, l=50, r=20),
)

fig.write_html(OUTPUT_DIR + "g21_ides.html")
fig.show(renderer="browser")

In [42]:
#G22: Education Level Distribution
label_curto = {
    "Primary/elementary school":                                                                     "Primary School",
    "Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)": "Secondary School",
    "Some college/university study without earning a degree":                                 "Some College",
    "Associate degree (A.A., A.S., etc.)": "Associate Degree",
    "Bachelor\u2019s degree (B.A., B.S., B.Eng., etc.)": "Bachelor's",
    "Master\u2019s degree (M.A., M.S., M.Eng., MBA, etc.)": "Master's",
    "Professional degree (JD, MD, Ph.D, Ed.D, etc.)": "Professional",
    "Something else": "Other",
}

ed_counts = (
    dataset["EdLevel"]
    .dropna()
    .pipe(lambda s: s[s != "Sem dado"])
    .value_counts()
    .reindex(list(label_curto.keys())).fillna(0)
    .reset_index()
)
ed_counts.columns    = ["EdLevel", "Count"]
ed_counts["Percent"] = (ed_counts["Count"] / ed_counts["Count"].sum() * 100).round(1)
ed_counts["Label"]   = ed_counts["EdLevel"].map(label_curto)

fig = go.Figure(go.Bar(
    x=ed_counts["Label"],
    y=ed_counts["Percent"],
    marker=dict(
        color=ed_counts["Percent"],
        colorscale="Blues",
        showscale=False,
        line=dict(color="white", width=1.5)
    ),
    text=[f'{p:.1f}%' for p in ed_counts["Percent"]],
    textposition="outside",
    hovertemplate="<b>%{x}</b><br>%{y:.1f}%<extra></extra>",
))
fig.update_layout(
    xaxis=dict(tickangle=-20, tickfont=dict(size=10), showgrid=False),
    yaxis=dict(
        title="% of Respondents",
        showgrid=True,
        gridcolor="#e5e7eb",
        range=[0, ed_counts["Percent"].max() + 10]
    ),
    paper_bgcolor="white",
    plot_bgcolor="white",
    margin=dict(t=60, b=100, l=50, r=20),
)

fig.write_html(OUTPUT_DIR + "g22_education.html")
fig.show(renderer="browser")

In [43]:
#G23: Job Satisfaction by Seniority Level
df_g6 = dataset.copy()
df_g6["YearsCodePro"] = pd.to_numeric(
    df_g6["YearsCodePro"].replace({"Less than 1 year": 0, "More than 50 years": 51}),
    errors="coerce")
df_g6["JobSat"] = pd.to_numeric(df_g6["JobSat"], errors="coerce")

def seniority(y):
    if pd.isna(y):  return None
    if y < 2:       return "Junior"
    if y < 6:       return "Mid"
    if y < 12:      return "Senior"
    return "Expert"

df_g6["Seniority"] = df_g6["YearsCodePro"].apply(seniority)
df_g6 = df_g6[df_g6["Seniority"].notna() & df_g6["JobSat"].notna()]

ordem = ["Junior", "Mid", "Senior", "Expert"]
cores = ["#1e88e5", "#14b8a6", "#fbbf24", "#fb7f3f"]

fig = go.Figure()
for nivel, cor in zip(ordem, cores):
    dados = df_g6[df_g6["Seniority"] == nivel]["JobSat"]
    fig.add_trace(go.Box(
        y=dados,
        name=nivel,
        marker_color=cor,
        boxmean=True,
        hovertemplate=f"<b>{nivel}</b><br>Job Satisfaction: %{{y}}<extra></extra>"
    ))
fig.update_layout(
    yaxis=dict(title="Job Satisfaction (0-10)", showgrid=True, gridcolor="#e5e7eb"),
    xaxis=dict(showgrid=False),
    paper_bgcolor="white",
    plot_bgcolor="white",
    margin=dict(t=60, b=40, l=50, r=20),
    showlegend=False,
)

fig.write_html(OUTPUT_DIR + "g23_jobsat_seniority.html")
fig.show(renderer="browser")

In [44]:
#G24: Salary Distribution by Work Modality
df_g7 = dataset[dataset["ConvertedCompYearly"] != 65000]
df_g7 = df_g7[["RemoteWork", "ConvertedCompYearly"]].dropna()
df_g7 = df_g7[df_g7["RemoteWork"] != "Sem dado"]
cap   = df_g7["ConvertedCompYearly"].quantile(0.95)
df_g7 = df_g7[df_g7["ConvertedCompYearly"] <= cap]

remote_order = ["Remote", "Hybrid (some remote, some in-person)", "In-person"]
cores_r      = ["#1e88e5", "#14b8a6", "#fbbf24"]

fig = go.Figure()
for mod, cor in zip(remote_order, cores_r):
    dados = df_g7[df_g7["RemoteWork"] == mod]["ConvertedCompYearly"]
    fig.add_trace(go.Box(
        y=dados,
        name=mod.replace("Hybrid (some remote, some in-person)", "Hybrid"),
        marker_color=cor,
        boxmean=True,
        hovertemplate="<b>%{fullData.name}</b><br>Salary: $%{y:,.0f}<extra></extra>"
    ))
fig.update_layout(
    yaxis=dict(title="Annual Salary (USD)", showgrid=True, gridcolor="#e5e7eb"),
    xaxis=dict(showgrid=False),
    paper_bgcolor="white",
    plot_bgcolor="white",
    margin=dict(t=60, b=40, l=60, r=20),
    showlegend=False,
)

fig.write_html(OUTPUT_DIR + "g24_salary_remote.html")
fig.show(renderer="browser")

In [45]:
#G25: Work Modality by Country (Top 10)
df_g8 = dataset[["Country", "RemoteWork"]].dropna()
df_g8 = df_g8[(df_g8["Country"] != "Sem dado") & (df_g8["RemoteWork"] != "Sem dado")]

top_paises = df_g8["Country"].value_counts().head(10).index
df_g8 = df_g8[df_g8["Country"].isin(top_paises)]

remote_pct = (df_g8.groupby(["Country", "RemoteWork"])
              .size().unstack(fill_value=0))
remote_pct = remote_pct.div(remote_pct.sum(axis=1), axis=0) * 100
remote_pct = remote_pct.round(1)

name_map = {
    "United States of America":                                              "United States",
    "United Kingdom of Great Britain and Northern Ireland": "United Kingdom",
}
remote_pct.index = [name_map.get(c, c) for c in remote_pct.index]

col_map = {
    "Remote":                                  "#1e88e5",
    "Hybrid (some remote, some in-person)": "#14b8a6",
    "In-person":                               "#fbbf24",
}
label_map = {
    "Remote":                                  "Remote",
    "Hybrid (some remote, some in-person)": "Hybrid",
    "In-person":                               "In-person",
}

fig = go.Figure()
for mod in remote_pct.columns:
    if mod in col_map:
        fig.add_trace(go.Bar(
            name=label_map.get(mod, mod),
            x=remote_pct.index,
            y=remote_pct[mod],
            marker_color=col_map[mod],
            hovertemplate="<b>%{x}</b><br>" + label_map.get(mod, mod) + ": %{y:.1f}%<extra></extra>"
        ))
fig.update_layout(
    barmode="stack",
    xaxis=dict(
        tickangle=-35,
        tickfont=dict(size=10),
        showgrid=False,
        automargin=True,
    ),
    yaxis=dict(title="% of Respondents", showgrid=True, gridcolor="#e5e7eb"),
    paper_bgcolor="white",
    plot_bgcolor="white",
    margin=dict(t=60, b=180, l=50, r=20),
    legend=dict(orientation="h", yanchor="bottom", y=-0.42, xanchor="center", x=0.5),
)

fig.write_html(OUTPUT_DIR + "g25_remote_by_country.html")
fig.show(renderer="browser")